In [ ]:

import pandas as pd
import numpy as np
import joblib

from IPython.display import display

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [ ]:
# ============================================================
#  LOAD CLEANED DATA
# ============================================================

DATA_PATH = "/content/Nassau_Candy_Cleaned.csv"

df = pd.read_csv(DATA_PATH)

print("======================================")
print("DATASET LOADED")
print("======================================")

print("Rows    :", len(df))
print("Columns :", len(df.columns))

display(df.head())

DATASET LOADED
Rows    : 10194
Columns : 25


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,...,Units,Gross Profit,Cost,Lead_Time,Profit Margin,Cost Percentage,Unit Price,Factory,Factory_Latitude,Factory_Longitude
0,1,US-2021-103800-CHO-MIL-31000,01-03-2024,30-06-2026,Standard Class,103800,United States,Houston,Texas,77095,...,2,4.22,2.28,851.0,0.649231,0.350769,3.25,Wicked Choccy's,32.076176,-81.088371
1,2,US-2021-112326-CHO-TRI-54000,01-04-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,...,2,4.90,2.60,821.0,0.653333,0.346667,3.75,Wicked Choccy's,32.076176,-81.088371
2,3,US-2021-112326-CHO-NUT-13000,01-04-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,...,3,7.47,3.00,821.0,0.713467,0.286533,3.49,Lot's O' Nuts,32.881893,-111.768036
3,4,US-2021-112326-CHO-SCR-58000,01-04-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,...,3,7.50,3.30,821.0,0.694444,0.305556,3.60,Lot's O' Nuts,32.881893,-111.768036
4,5,US-2021-141817-CHO-TRI-54000,01-05-2024,05-07-2026,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,...,3,7.35,3.90,795.0,0.653333,0.346667,3.75,Wicked Choccy's,32.076176,-81.088371


In [ ]:
# ============================================================
#  CHECK ORIGINAL COLUMNS
# ============================================================

required_original_columns = [
    "Order ID",
    "Order Date",
    "Product ID",
    "Product Name",
    "Division",
    "Region",
    "State/Province",
    "Ship Mode",
    "Factory",
    "Sales",
    "Units",
    "Cost",
    "Gross Profit"
]

missing_columns = [
    col for col in required_original_columns
    if col not in df.columns
]

if len(missing_columns) == 0:
    print("All required original columns are available!")
else:
    print("Missing columns:")
    print(missing_columns)

All required original columns are available!


In [ ]:
# ============================================================
#  LOAD WORKING RANDOM FOREST
# ============================================================

MODEL_PATH = "/content/best_model_random_forest.pkl"

model = joblib.load(MODEL_PATH)

print("======================================")
print("RECOMMENDATION MODEL LOADED")
print("======================================")

print(type(model))
print(model)

RECOMMENDATION MODEL LOADED
<class 'sklearn.pipeline.Pipeline'>
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Product ID', 'Product Name',
                                                   'Division', 'Region',
                                                   'State/Province',
                                                   'Ship Mode', 'Factory']),
                                                 ('numerical',
                                                  Pipeline(steps=[('imputer',
                      

In [ ]:
# ============================================================
#  CREATE MODEL DATAFRAME
# ============================================================

model_df = df.copy()

print("Model dataframe created.")

print("Rows:", len(model_df))

Model dataframe created.
Rows: 10194


In [ ]:
# ============================================================
#  CONVERT ORDER DATE
# ============================================================

model_df["Order Date"] = pd.to_datetime(
    model_df["Order Date"],
    errors="coerce"
)

print("Order Date converted successfully.")

print(
    "Invalid Order Dates:",
    model_df["Order Date"].isna().sum()
)

Order Date converted successfully.
Invalid Order Dates: 6064


In [ ]:
# ============================================================
#  CREATE DATE FEATURES
# ============================================================

model_df["Order Year"] = (
    model_df["Order Date"].dt.year
)

model_df["Order Month"] = (
    model_df["Order Date"].dt.month
)

model_df["Order Quarter"] = (
    model_df["Order Date"].dt.quarter
)

model_df["Order Day of Week"] = (
    model_df["Order Date"].dt.dayofweek
)

print("Date features created!")

display(
    model_df[
        [
            "Order Date",
            "Order Year",
            "Order Month",
            "Order Quarter",
            "Order Day of Week"
        ]
    ].head()
)

Date features created!


,Order Date,Order Year,Order Month,Order Quarter,Order Day of Week
0,2024-01-03,2024.0,1.0,1.0,2.0
1,2024-01-04,2024.0,1.0,1.0,3.0
2,2024-01-04,2024.0,1.0,1.0,3.0
3,2024-01-04,2024.0,1.0,1.0,3.0
4,2024-01-05,2024.0,1.0,1.0,4.0


In [ ]:
# ============================================================
#  CREATE FINANCIAL FEATURES
# ============================================================

model_df["Profit Margin"] = np.where(
    model_df["Sales"] != 0,
    model_df["Gross Profit"] / model_df["Sales"],
    0
)

model_df["Cost Percentage"] = np.where(
    model_df["Sales"] != 0,
    model_df["Cost"] / model_df["Sales"],
    0
)

model_df["Unit Price"] = np.where(
    model_df["Units"] != 0,
    model_df["Sales"] / model_df["Units"],
    0
)

print("Financial features created!")

display(
    model_df[
        [
            "Sales",
            "Units",
            "Cost",
            "Gross Profit",
            "Profit Margin",
            "Cost Percentage",
            "Unit Price"
        ]
    ].head()
)

Financial features created!


,Sales,Units,Cost,Gross Profit,Profit Margin,Cost Percentage,Unit Price
0,6.50,2,2.28,4.22,0.649231,0.350769,3.25
1,7.50,2,2.60,4.90,0.653333,0.346667,3.75
2,10.47,3,3.00,7.47,0.713467,0.286533,3.49
3,10.80,3,3.30,7.50,0.694444,0.305556,3.60
4,11.25,3,3.90,7.35,0.653333,0.346667,3.75


In [ ]:
# ============================================================
# CELL 9 — MODEL FEATURE CHECK
# ============================================================

model_features = [
    "Product ID",
    "Product Name",
    "Division",
    "Region",
    "State/Province",
    "Ship Mode",
    "Factory",
    "Sales",
    "Units",
    "Cost",
    "Gross Profit",
    "Profit Margin",
    "Cost Percentage",
    "Unit Price",
    "Order Year",
    "Order Month",
    "Order Quarter",
    "Order Day of Week"
]

missing_features = [
    col for col in model_features
    if col not in model_df.columns
]

print("======================================")
print("MODEL FEATURE CHECK")
print("======================================")

if len(missing_features) == 0:
    print("All 18 model features are available!")
else:
    print("Missing features:")
    print(missing_features)

MODEL FEATURE CHECK
All 18 model features are available!


In [ ]:
# ============================================================
# CELL 10 — FACTORIES
# ============================================================

factories = [
    "Lot's O' Nuts",
    "Wicked Choccy's",
    "Sugar Shack",
    "Secret Factory",
    "The Other Factory"
]

print("Factories available:")
print(factories)

Factories available:
["Lot's O' Nuts", "Wicked Choccy's", 'Sugar Shack', 'Secret Factory', 'The Other Factory']


In [ ]:
# ============================================================
# CELL 11 — CREATE FACTORY SCENARIOS
# ============================================================

scenario_rows = []

for _, row in model_df.iterrows():

    for factory in factories:

        scenario = row.copy()

        scenario["Current Factory"] = row["Factory"]

        # Change factory for simulation
        scenario["Factory"] = factory

        scenario_rows.append(scenario)

scenario_df = pd.DataFrame(scenario_rows)

print("======================================")
print("SCENARIO DATASET CREATED")
print("======================================")

print("Original rows :", len(model_df))
print("Scenario rows :", len(scenario_df))

display(
    scenario_df[
        [
            "Order ID",
            "Product Name",
            "Current Factory",
            "Factory"
        ]
    ].head(15)
)

SCENARIO DATASET CREATED
Original rows : 10194
Scenario rows : 50970


,Order ID,Product Name,Current Factory,Factory
0,US-2021-103800-CHO-MIL-31000,Wonka Bar - Milk Chocolate,Wicked Choccy's,Lot's O' Nuts
0,US-2021-103800-CHO-MIL-31000,Wonka Bar - Milk Chocolate,Wicked Choccy's,Wicked Choccy's
0,US-2021-103800-CHO-MIL-31000,Wonka Bar - Milk Chocolate,Wicked Choccy's,Sugar Shack
0,US-2021-103800-CHO-MIL-31000,Wonka Bar - Milk Chocolate,Wicked Choccy's,Secret Factory
0,US-2021-103800-CHO-MIL-31000,Wonka Bar - Milk Chocolate,Wicked Choccy's,The Other Factory
1,US-2021-112326-CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,Lot's O' Nuts
1,US-2021-112326-CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,Wicked Choccy's
1,US-2021-112326-CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,Sugar Shack
1,US-2021-112326-CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,Secret Factory
1,US-2021-112326-CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,The Other Factory


In [ ]:
# ============================================================
# CELL 12 — PREPARE PREDICTION DATA
# ============================================================

X_scenario = scenario_df[model_features].copy()

print("Prediction dataset created.")

print(
    "Prediction rows   :",
    len(X_scenario)
)

print(
    "Prediction columns:",
    len(X_scenario.columns)
)

display(X_scenario.head())

Prediction dataset created.
Prediction rows   : 50970
Prediction columns: 18


,Product ID,Product Name,Division,Region,State/Province,Ship Mode,Factory,Sales,Units,Cost,Gross Profit,Profit Margin,Cost Percentage,Unit Price,Order Year,Order Month,Order Quarter,Order Day of Week
0,CHO-MIL-31000,Wonka Bar - Milk Chocolate,Chocolate,Interior,Texas,Standard Class,Lot's O' Nuts,6.5,2,2.28,4.22,0.649231,0.350769,3.25,2024.0,1.0,1.0,2.0
0,CHO-MIL-31000,Wonka Bar - Milk Chocolate,Chocolate,Interior,Texas,Standard Class,Wicked Choccy's,6.5,2,2.28,4.22,0.649231,0.350769,3.25,2024.0,1.0,1.0,2.0
0,CHO-MIL-31000,Wonka Bar - Milk Chocolate,Chocolate,Interior,Texas,Standard Class,Sugar Shack,6.5,2,2.28,4.22,0.649231,0.350769,3.25,2024.0,1.0,1.0,2.0
0,CHO-MIL-31000,Wonka Bar - Milk Chocolate,Chocolate,Interior,Texas,Standard Class,Secret Factory,6.5,2,2.28,4.22,0.649231,0.350769,3.25,2024.0,1.0,1.0,2.0
0,CHO-MIL-31000,Wonka Bar - Milk Chocolate,Chocolate,Interior,Texas,Standard Class,The Other Factory,6.5,2,2.28,4.22,0.649231,0.350769,3.25,2024.0,1.0,1.0,2.0


In [ ]:
# ============================================================
# CELL 13 — PREDICT LEAD TIME
# ============================================================

print("Generating predictions...")

scenario_df["Predicted Lead_Time"] = model.predict(
    X_scenario
)

print("======================================")
print("PREDICTIONS COMPLETED")
print("======================================")

print(
    "Minimum predicted lead time:",
    scenario_df["Predicted Lead_Time"].min()
)

print(
    "Maximum predicted lead time:",
    scenario_df["Predicted Lead_Time"].max()
)

print(
    "Average predicted lead time:",
    scenario_df["Predicted Lead_Time"].mean()
)

display(
    scenario_df[
        [
            "Order ID",
            "Product Name",
            "Current Factory",
            "Factory",
            "Predicted Lead_Time"
        ]
    ].head(20)
)

Generating predictions...
PREDICTIONS COMPLETED
Minimum predicted lead time: 655.3966666666666
Maximum predicted lead time: 1947.17
Average predicted lead time: 1486.351334276024


,Order ID,Product Name,Current Factory,Factory,Predicted Lead_Time
0,US-2021-103800-CHO-MIL-31000,Wonka Bar - Milk Chocolate,Wicked Choccy's,Lot's O' Nuts,1028.811762
0,US-2021-103800-CHO-MIL-31000,Wonka Bar - Milk Chocolate,Wicked Choccy's,Wicked Choccy's,1014.502429
0,US-2021-103800-CHO-MIL-31000,Wonka Bar - Milk Chocolate,Wicked Choccy's,Sugar Shack,1023.369095
0,US-2021-103800-CHO-MIL-31000,Wonka Bar - Milk Chocolate,Wicked Choccy's,Secret Factory,1018.144095
0,US-2021-103800-CHO-MIL-31000,Wonka Bar - Milk Chocolate,Wicked Choccy's,The Other Factory,1023.369095
1,US-2021-112326-CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,Lot's O' Nuts,861.090000
1,US-2021-112326-CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,Wicked Choccy's,856.516667
1,US-2021-112326-CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,Sugar Shack,860.146667
1,US-2021-112326-CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,Secret Factory,859.690000
1,US-2021-112326-CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,The Other Factory,861.750000


In [ ]:
# ============================================================
# CELL 14 — BEST FACTORY FOR EACH ORDER
# ============================================================

best_factory_df = (
    scenario_df
    .sort_values(
        ["Order ID", "Predicted Lead_Time"]
    )
    .groupby("Order ID", as_index=False)
    .first()
)

best_factory_df = best_factory_df.rename(
    columns={
        "Factory": "Recommended Factory",
        "Predicted Lead_Time":
            "Recommended Predicted Lead_Time"
    }
)

print("======================================")
print("BEST FACTORY IDENTIFIED")
print("======================================")

display(
    best_factory_df[
        [
            "Order ID",
            "Product Name",
            "Current Factory",
            "Recommended Factory",
            "Recommended Predicted Lead_Time"
        ]
    ].head(20)
)

BEST FACTORY IDENTIFIED


,Order ID,Product Name,Current Factory,Recommended Factory,Recommended Predicted Lead_Time
0,CA-2021-100867-CHO-FUD-51000,Wonka Bar - Fudge Mallows,Lot's O' Nuts,The Other Factory,1555.342667
1,CA-2021-107153-CHO-FUD-51000,Wonka Bar - Fudge Mallows,Lot's O' Nuts,Lot's O' Nuts,1620.503810
2,CA-2021-115238-CHO-FUD-51000,Wonka Bar - Fudge Mallows,Lot's O' Nuts,Lot's O' Nuts,1597.785556
3,CA-2021-115238-CHO-MIL-31000,Wonka Bar - Milk Chocolate,Wicked Choccy's,Wicked Choccy's,1510.370000
4,CA-2021-115238-CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,Wicked Choccy's,1632.273667
5,CA-2021-115777-CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts,Lot's O' Nuts,1572.659667
6,CA-2021-117964-CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts,Sugar Shack,1379.270556
7,CA-2021-117964-CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,Lot's O' Nuts,1372.023333
8,CA-2021-119508-CHO-FUD-51000,Wonka Bar - Fudge Mallows,Lot's O' Nuts,Lot's O' Nuts,1243.670000
9,CA-2021-119508-CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts,Lot's O' Nuts,1218.620000


In [ ]:
# ============================================================
# CELL 15 — CURRENT FACTORY PREDICTION
# ============================================================

current_factory_df = scenario_df[
    scenario_df["Factory"]
    ==
    scenario_df["Current Factory"]
].copy()

current_factory_df = (
    current_factory_df
    .drop_duplicates("Order ID")
)

current_predictions = (
    current_factory_df
    .set_index("Order ID")[
        "Predicted Lead_Time"
    ]
)

best_factory_df["Current Predicted Lead_Time"] = (
    best_factory_df["Order ID"]
    .map(current_predictions)
)

print("Current factory predictions added.")

display(
    best_factory_df[
        [
            "Order ID",
            "Product Name",
            "Current Factory",
            "Current Predicted Lead_Time",
            "Recommended Factory",
            "Recommended Predicted Lead_Time"
        ]
    ].head(20)
)

Current factory predictions added.


,Order ID,Product Name,Current Factory,Current Predicted Lead_Time,Recommended Factory,Recommended Predicted Lead_Time
0,CA-2021-100867-CHO-FUD-51000,Wonka Bar - Fudge Mallows,Lot's O' Nuts,1556.822667,The Other Factory,1555.342667
1,CA-2021-107153-CHO-FUD-51000,Wonka Bar - Fudge Mallows,Lot's O' Nuts,1620.503810,Lot's O' Nuts,1620.503810
2,CA-2021-115238-CHO-FUD-51000,Wonka Bar - Fudge Mallows,Lot's O' Nuts,1659.623333,Lot's O' Nuts,1597.785556
3,CA-2021-115238-CHO-MIL-31000,Wonka Bar - Milk Chocolate,Wicked Choccy's,1686.050000,Wicked Choccy's,1510.370000
4,CA-2021-115238-CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,1632.273667,Wicked Choccy's,1632.273667
5,CA-2021-115777-CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts,1572.659667,Lot's O' Nuts,1572.659667
6,CA-2021-117964-CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts,1379.513889,Sugar Shack,1379.270556
7,CA-2021-117964-CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,1375.287500,Lot's O' Nuts,1372.023333
8,CA-2021-119508-CHO-FUD-51000,Wonka Bar - Fudge Mallows,Lot's O' Nuts,1243.670000,Lot's O' Nuts,1243.670000
9,CA-2021-119508-CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts,1218.620000,Lot's O' Nuts,1218.620000


In [ ]:
# ============================================================
# CELL 16 — LEAD TIME REDUCTION
# ============================================================

best_factory_df["Lead_Time Reduction"] = (
    best_factory_df["Current Predicted Lead_Time"]
    -
    best_factory_df["Recommended Predicted Lead_Time"]
)

best_factory_df["Lead_Time Reduction (%)"] = np.where(
    best_factory_df["Current Predicted Lead_Time"] > 0,

    (
        best_factory_df["Lead_Time Reduction"]
        /
        best_factory_df["Current Predicted Lead_Time"]
    ) * 100,

    0
)

print("======================================")
print("CURRENT VS RECOMMENDED PERFORMANCE")
print("======================================")

display(
    best_factory_df[
        [
            "Order ID",
            "Product Name",
            "Current Factory",
            "Recommended Factory",
            "Current Predicted Lead_Time",
            "Recommended Predicted Lead_Time",
            "Lead_Time Reduction",
            "Lead_Time Reduction (%)"
        ]
    ].head(20)
)

CURRENT VS RECOMMENDED PERFORMANCE


,Order ID,Product Name,Current Factory,Recommended Factory,Current Predicted Lead_Time,Recommended Predicted Lead_Time,Lead_Time Reduction,Lead_Time Reduction (%)
0,CA-2021-100867-CHO-FUD-51000,Wonka Bar - Fudge Mallows,Lot's O' Nuts,The Other Factory,1556.822667,1555.342667,1.480000,0.095065
1,CA-2021-107153-CHO-FUD-51000,Wonka Bar - Fudge Mallows,Lot's O' Nuts,Lot's O' Nuts,1620.503810,1620.503810,0.000000,0.000000
2,CA-2021-115238-CHO-FUD-51000,Wonka Bar - Fudge Mallows,Lot's O' Nuts,Lot's O' Nuts,1659.623333,1597.785556,61.837778,3.726013
3,CA-2021-115238-CHO-MIL-31000,Wonka Bar - Milk Chocolate,Wicked Choccy's,Wicked Choccy's,1686.050000,1510.370000,175.680000,10.419620
4,CA-2021-115238-CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,Wicked Choccy's,1632.273667,1632.273667,0.000000,0.000000
5,CA-2021-115777-CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts,Lot's O' Nuts,1572.659667,1572.659667,0.000000,0.000000
6,CA-2021-117964-CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts,Sugar Shack,1379.513889,1379.270556,0.243333,0.017639
7,CA-2021-117964-CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,Lot's O' Nuts,1375.287500,1372.023333,3.264167,0.237344
8,CA-2021-119508-CHO-FUD-51000,Wonka Bar - Fudge Mallows,Lot's O' Nuts,Lot's O' Nuts,1243.670000,1243.670000,0.000000,0.000000
9,CA-2021-119508-CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts,Lot's O' Nuts,1218.620000,1218.620000,0.000000,0.000000


In [ ]:
# ============================================================
# CELL 17 — RECOMMENDATION LOGIC
# ============================================================

MIN_IMPROVEMENT_PCT = 0.10

best_factory_df["Recommendation"] = np.where(
    (
        (best_factory_df["Recommended Factory"]
         != best_factory_df["Current Factory"])
        &
        (best_factory_df["Lead_Time Reduction (%)"]
         >= MIN_IMPROVEMENT_PCT)
    ),

    "Reassign",

    "Keep Current Factory"
)

print("======================================")
print("RECOMMENDATION DISTRIBUTION")
print("======================================")

print(
    best_factory_df["Recommendation"]
    .value_counts()
)

RECOMMENDATION DISTRIBUTION
Recommendation
Keep Current Factory    4386
Reassign                4163
Name: count, dtype: int64


In [ ]:
# ============================================================
# CELL 18 — RECOMMENDED FACTORY DISTRIBUTION
# ============================================================

print("======================================")
print("RECOMMENDED FACTORY DISTRIBUTION")
print("======================================")

print(
    best_factory_df.loc[
        best_factory_df["Recommendation"]
        == "Reassign",
        "Recommended Factory"
    ].value_counts()
)

print("\nCurrent Factory Distribution")

print(
    best_factory_df["Current Factory"]
    .value_counts()
)

RECOMMENDED FACTORY DISTRIBUTION
Recommended Factory
Wicked Choccy's      1602
Lot's O' Nuts        1213
The Other Factory     503
Secret Factory        465
Sugar Shack           380
Name: count, dtype: int64

Current Factory Distribution
Current Factory
Lot's O' Nuts        4760
Wicked Choccy's      3445
Secret Factory        213
The Other Factory      98
Sugar Shack            33
Name: count, dtype: int64


In [ ]:
# ============================================================
# CELL 19 — PRODUCT LEVEL RECOMMENDATION TABLE
# ============================================================

product_recommendations = (
    best_factory_df
    .groupby(
        [
            "Product ID",
            "Product Name",
            "Current Factory"
        ],
        as_index=False
    )
    .agg(
        Current_Predicted_Lead_Time=(
            "Current Predicted Lead_Time",
            "mean"
        ),

        Recommended_Predicted_Lead_Time=(
            "Recommended Predicted Lead_Time",
            "mean"
        ),

        Lead_Time_Reduction=(
            "Lead_Time Reduction",
            "mean"
        ),

        Lead_Time_Reduction_Pct=(
            "Lead_Time Reduction (%)",
            "mean"
        )
    )
)

# Most frequently recommended factory
recommended_factory = (
    best_factory_df
    .groupby(
        [
            "Product ID",
            "Product Name",
            "Current Factory"
        ]
    )["Recommended Factory"]
    .agg(
        lambda x: x.value_counts().index[0]
    )
    .reset_index()
)

product_recommendations = (
    product_recommendations
    .merge(
        recommended_factory,
        on=[
            "Product ID",
            "Product Name",
            "Current Factory"
        ],
        how="left"
    )
)

product_recommendations["Recommendation"] = np.where(
    (
        (product_recommendations["Recommended Factory"]
         != product_recommendations["Current Factory"])
        &
        (product_recommendations["Lead_Time_Reduction_Pct"]
         >= MIN_IMPROVEMENT_PCT)
    ),

    "Reassign",

    "Keep Current Factory"
)

print("======================================")
print("PRODUCT-LEVEL RECOMMENDATION TABLE")
print("======================================")

print(
    "Unique products evaluated:",
    product_recommendations["Product ID"].nunique()
)

display(product_recommendations)

PRODUCT-LEVEL RECOMMENDATION TABLE
Unique products evaluated: 15


,Product ID,Product Name,Current Factory,Current_Predicted_Lead_Time,Recommended_Predicted_Lead_Time,Lead_Time_Reduction,Lead_Time_Reduction_Pct,Recommended Factory,Recommendation
0,CHO-FUD-51000,Wonka Bar - Fudge Mallows,Lot's O' Nuts,1501.288572,1493.698471,7.590101,0.495477,Wicked Choccy's,Reassign
1,CHO-MIL-31000,Wonka Bar - Milk Chocolate,Wicked Choccy's,1501.475485,1492.576541,8.898944,0.575485,Lot's O' Nuts,Reassign
2,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts,1451.250896,1445.882954,5.367942,0.365547,Lot's O' Nuts,Keep Current Factory
3,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts,1486.532242,1479.608521,6.923720,0.455563,Lot's O' Nuts,Keep Current Factory
4,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,1485.988902,1478.870846,7.118057,0.468852,Wicked Choccy's,Keep Current Factory
5,OTH-FIZ-56000,Fizzy Lifting Drinks,Sugar Shack,1546.228981,1545.004907,1.224074,0.079380,Wicked Choccy's,Keep Current Factory
6,OTH-GUM-21000,Wonka Gum,Secret Factory,1487.324428,1482.041867,5.282561,0.340248,Lot's O' Nuts,Reassign
7,OTH-KAZ-38000,Kazookles,The Other Factory,1502.202007,1499.897729,2.304278,0.153968,Lot's O' Nuts,Reassign
8,OTH-LIC-15000,Lickable Wallpaper,Secret Factory,1495.085925,1489.694926,5.390999,0.347561,Wicked Choccy's,Reassign
9,SUG-EVE-47000,Everlasting Gobstopper,Secret Factory,1501.071463,1498.956648,2.114815,0.135597,Secret Factory,Keep Current Factory


In [ ]:
# ============================================================
# CELL 20 — RECOMMENDATION COVERAGE
# ============================================================

total_products = (
    product_recommendations["Product ID"]
    .nunique()
)

recommended_products = (
    product_recommendations.loc[
        product_recommendations["Recommendation"]
        == "Reassign",
        "Product ID"
    ]
    .nunique()
)

recommendation_coverage = (
    recommended_products
    /
    total_products
) * 100

print("======================================")
print("RECOMMENDATION COVERAGE")
print("======================================")

print(
    "Total Products Evaluated :",
    total_products
)

print(
    "Products Recommended     :",
    recommended_products
)

print(
    f"Recommendation Coverage  : "
    f"{recommendation_coverage:.2f}%"
)

RECOMMENDATION COVERAGE
Total Products Evaluated : 15
Products Recommended     : 8
Recommendation Coverage  : 53.33%


In [ ]:
# ============================================================
# CELL 21 — KPI SUMMARY
# ============================================================

total_orders = best_factory_df["Order ID"].nunique()

reassigned_orders = (
    best_factory_df["Recommendation"]
    == "Reassign"
).sum()

avg_current_lead_time = (
    best_factory_df["Current Predicted Lead_Time"]
    .mean()
)

avg_recommended_lead_time = (
    best_factory_df["Recommended Predicted Lead_Time"]
    .mean()
)

overall_reduction = (
    avg_current_lead_time
    -
    avg_recommended_lead_time
)

overall_reduction_pct = (
    overall_reduction
    /
    avg_current_lead_time
) * 100

print("======================================")
print("RECOMMENDATION ENGINE KPI SUMMARY")
print("======================================")

print(f"Total Orders Evaluated      : {total_orders}")
print(f"Orders Recommended          : {reassigned_orders}")
print(f"Total Products Evaluated    : {total_products}")
print(f"Products Recommended        : {recommended_products}")
print(f"Recommendation Coverage     : {recommendation_coverage:.2f}%")
print(f"Average Current Lead Time   : {avg_current_lead_time:.2f}")
print(f"Average Recommended Lead Time: {avg_recommended_lead_time:.2f}")
print(f"Average Lead Time Reduction : {overall_reduction:.2f}")
print(f"Overall Reduction (%)       : {overall_reduction_pct:.2f}%")

RECOMMENDATION ENGINE KPI SUMMARY
Total Orders Evaluated      : 8549
Orders Recommended          : 4163
Total Products Evaluated    : 15
Products Recommended        : 8
Recommendation Coverage     : 53.33%
Average Current Lead Time   : 1486.41
Average Recommended Lead Time: 1479.30
Average Lead Time Reduction : 7.11
Overall Reduction (%)       : 0.48%


In [ ]:
# ============================================================
# CELL 22 — TOP RECOMMENDATIONS
# ============================================================

top_recommendations = (
    product_recommendations[
        product_recommendations["Recommendation"]
        == "Reassign"
    ]
    .sort_values(
        "Lead_Time_Reduction_Pct",
        ascending=False
    )
)

print("======================================")
print("TOP PRODUCT RECOMMENDATIONS")
print("======================================")

display(
    top_recommendations[
        [
            "Product ID",
            "Product Name",
            "Current Factory",
            "Recommended Factory",
            "Current_Predicted_Lead_Time",
            "Recommended_Predicted_Lead_Time",
            "Lead_Time_Reduction",
            "Lead_Time_Reduction_Pct"
        ]
    ].head(20)
)

TOP PRODUCT RECOMMENDATIONS


,Product ID,Product Name,Current Factory,Recommended Factory,Current_Predicted_Lead_Time,Recommended_Predicted_Lead_Time,Lead_Time_Reduction,Lead_Time_Reduction_Pct
14,SUG-SWE-91000,SweeTARTS,Sugar Shack,Wicked Choccy's,1603.946578,1594.560522,9.386056,0.582929
1,CHO-MIL-31000,Wonka Bar - Milk Chocolate,Wicked Choccy's,Lot's O' Nuts,1501.475485,1492.576541,8.898944,0.575485
0,CHO-FUD-51000,Wonka Bar - Fudge Mallows,Lot's O' Nuts,Wicked Choccy's,1501.288572,1493.698471,7.590101,0.495477
8,OTH-LIC-15000,Lickable Wallpaper,Secret Factory,Wicked Choccy's,1495.085925,1489.694926,5.390999,0.347561
6,OTH-GUM-21000,Wonka Gum,Secret Factory,Lot's O' Nuts,1487.324428,1482.041867,5.282561,0.340248
12,SUG-LAF-25000,Laffy Taffy,Sugar Shack,Lot's O' Nuts,1603.782433,1599.903878,3.878556,0.237717
13,SUG-NER-92000,Nerds,Sugar Shack,Lot's O' Nuts,1435.310556,1432.261806,3.048750,0.209205
7,OTH-KAZ-38000,Kazookles,The Other Factory,Lot's O' Nuts,1502.202007,1499.897729,2.304278,0.153968


In [ ]:
# ============================================================
# CELL 23 — EXPORT RESULTS
# ============================================================

scenario_df.to_csv(
    "/content/scenario_simulation.csv",
    index=False
)

best_factory_df.to_csv(
    "/content/order_recommendations.csv",
    index=False
)

product_recommendations.to_csv(
    "/content/product_recommendations.csv",
    index=False
)

print("======================================")
print("FILES SAVED SUCCESSFULLY")
print("======================================")

print("/content/scenario_simulation.csv")
print("/content/order_recommendations.csv")
print("/content/product_recommendations.csv")

FILES SAVED SUCCESSFULLY
/content/scenario_simulation.csv
/content/order_recommendations.csv
/content/product_recommendations.csv


In [ ]:
# ============================================================
# CELL 24 — FINAL VERIFICATION
# ============================================================

print("======================================")
print("RECOMMENDATION ENGINE COMPLETED")
print("======================================")

print(
    "Model:",
    "Random Forest Regressor"
)

print(
    "Scenario rows:",
    len(scenario_df)
)

print(
    "Orders evaluated:",
    total_orders
)

print(
    "Products evaluated:",
    total_products
)

print(
    "Products recommended:",
    recommended_products
)

print(
    f"Recommendation Coverage: "
    f"{recommendation_coverage:.2f}%"
)

print(
    f"Overall Lead Time Reduction: "
    f"{overall_reduction_pct:.2f}%"
)

print("\nOutput files:")
print("1. scenario_simulation.csv")
print("2. order_recommendations.csv")
print("3. product_recommendations.csv")

RECOMMENDATION ENGINE COMPLETED
Model: Random Forest Regressor
Scenario rows: 50970
Orders evaluated: 8549
Products evaluated: 15
Products recommended: 8
Recommendation Coverage: 53.33%
Overall Lead Time Reduction: 0.48%

Output files:
1. scenario_simulation.csv
2. order_recommendations.csv
3. product_recommendations.csv


In [ ]:
# ======================================
# EXPORT FINAL RECOMMENDATION OUTPUTS
# ======================================

import os

output_dir = "/content/recommendation_outputs"
os.makedirs(output_dir, exist_ok=True)

# 1. Order-level recommendations
best_factory_df.to_csv(
    f"{output_dir}/order_recommendations.csv",
    index=False
)

# 2. Product-level recommendations
product_recommendations.to_csv(
    f"{output_dir}/product_recommendations.csv",
    index=False
)

# 3. Reassignment recommendations only
reassignment_df = best_factory_df[
    best_factory_df["Recommendation"] == "Reassign"
].copy()

reassignment_df.to_csv(
    f"{output_dir}/reassignment_recommendations.csv",
    index=False
)

print("=" * 50)
print("RECOMMENDATION FILES EXPORTED")
print("=" * 50)

print("\nOutput folder:")
print(output_dir)

print("\nFiles created:")
print("1. order_recommendations.csv")
print("2. product_recommendations.csv")
print("3. reassignment_recommendations.csv")

print("\nRows:")
print("Order recommendations      :", len(best_factory_df))
print("Product recommendations    :", len(product_recommendations))
print("Reassignment recommendations:", len(reassignment_df))

RECOMMENDATION FILES EXPORTED

Output folder:
/content/recommendation_outputs

Files created:
1. order_recommendations.csv
2. product_recommendations.csv
3. reassignment_recommendations.csv

Rows:
Order recommendations      : 8549
Product recommendations    : 15
Reassignment recommendations: 4163


In [ ]:
# ======================================
# RECOMMENDATION SUMMARY FOR POWER BI
# ======================================

recommendation_summary = pd.DataFrame({
    "Metric": [
        "Total Products Evaluated",
        "Products Recommended",
        "Recommendation Coverage"
    ],
    "Value": [
        15,
        8,
        53.33
    ]
})

display(recommendation_summary)

# Export
recommendation_summary.to_csv(
    "/content/recommendation_outputs/recommendation_summary.csv",
    index=False
)

print("Recommendation summary exported successfully!")

,Metric,Value
0,Total Products Evaluated,15.00
1,Products Recommended,8.00
2,Recommendation Coverage,53.33


Recommendation summary exported successfully!


In [ ]:
print("Actual Lead_Time statistics")
print("--------------------------------")

print("Minimum:", df["Lead_Time"].min())
print("Maximum:", df["Lead_Time"].max())
print("Mean:", df["Lead_Time"].mean())
print("Median:", df["Lead_Time"].median())

print("\nPercentiles:")
print(df["Lead_Time"].quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]))

Actual Lead_Time statistics
--------------------------------
Minimum: 613.0
Maximum: 1965.0
Mean: 1366.0096852300242
Median: 1361.0

Percentiles:
0.25    1155.0
0.50    1361.0
0.75    1598.0
0.90    1785.0
0.95    1870.0
0.99    1932.0
Name: Lead_Time, dtype: float64


In [ ]:
print("\nSample actual lead times:")
display(
    df[
        [
            "Order Date",
            "Ship Date",
            "Lead_Time",
            "Product Name",
            "Factory"
        ]
    ].head(20)
)


Sample actual lead times:


,Order Date,Ship Date,Lead_Time,Product Name,Factory
0,01-03-2024,30-06-2026,851.0,Wonka Bar - Milk Chocolate,Wicked Choccy's
1,01-04-2024,01-07-2026,821.0,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's
2,01-04-2024,01-07-2026,821.0,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts
3,01-04-2024,01-07-2026,821.0,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts
4,01-05-2024,05-07-2026,795.0,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's
5,01-06-2024,03-07-2026,762.0,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts
6,01-06-2024,03-07-2026,762.0,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's
7,01-06-2024,30-06-2026,759.0,Wonka Bar - Milk Chocolate,Wicked Choccy's
8,01-06-2024,03-07-2026,762.0,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts
9,01-06-2024,03-07-2026,762.0,Wonka Bar - Milk Chocolate,Wicked Choccy's


In [ ]:
# ============================================================
# MODEL VALIDATION - FIX NaN ERROR
# ============================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

model_features = [
    "Product ID",
    "Product Name",
    "Division",
    "Region",
    "State/Province",
    "Ship Mode",
    "Factory",
    "Sales",
    "Units",
    "Cost",
    "Gross Profit",
    "Profit Margin",
    "Cost Percentage",
    "Unit Price",
    "Order Year",
    "Order Month",
    "Order Quarter",
    "Order Day of Week"
]

# Create validation dataset from model_df which contains engineered features
validation_df = model_df[model_features + ["Lead_Time"]].copy()

# Remove rows where actual target Lead_Time is missing
validation_df = validation_df.dropna(
    subset=["Lead_Time"]
).reset_index(drop=True)

X_check = validation_df[model_features]
y_actual = validation_df["Lead_Time"]

# Predict
y_pred = model.predict(X_check)

# Convert predictions to numpy array
y_pred = np.asarray(y_pred)

# Check for NaN / infinite predictions
valid_mask = (
    np.isfinite(y_actual.to_numpy()) &
    np.isfinite(y_pred)
)

# Keep only valid rows
y_actual_clean = y_actual.to_numpy()[valid_mask]
y_pred_clean = y_pred[valid_mask]

# Calculate metrics
mae = mean_absolute_error(
    y_actual_clean,
    y_pred_clean
)

rmse = np.sqrt(
    mean_squared_error(
        y_actual_clean,
        y_pred_clean
    )
)

r2 = r2_score(
    y_actual_clean,
    y_pred_clean
)

print("======================================")
print("MODEL VALIDATION")
print("======================================")

print(f"Valid observations : {len(y_actual_clean)}")
print(f"MAE                : {mae:.2f} days")
print(f"RMSE               : {rmse:.2f} days")
print(f"R²                 : {r2:.4f}")

print("\nActual Lead Time")
print("--------------------------------------")
print(f"Mean   : {y_actual_clean.mean():.2f} days")
print(f"Median : {np.median(y_actual_clean):.2f} days")
print(f"Min    : {y_actual_clean.min():.2f} days")
print(f"Max    : {y_actual_clean.max():.2f} days")

print("\nPredicted Lead Time")
print("--------------------------------------")
print(f"Mean   : {y_pred_clean.mean():.2f} days")
print(f"Median : {np.median(y_pred_clean):.2f} days")
print(f"Min    : {y_pred_clean.min():.2f} days")
print(f"Max    : {y_pred_clean.max():.2f} days")

print("\nRows removed because of invalid values:",
      len(y_actual) - len(y_actual_clean))

MODEL VALIDATION
Valid observations : 4130
MAE                : 70.67 days
RMSE               : 98.66 days
R²                 : 0.8938

Actual Lead Time
--------------------------------------
Mean   : 1366.01 days
Median : 1361.00 days
Min    : 613.00 days
Max    : 1965.00 days

Predicted Lead Time
--------------------------------------
Mean   : 1368.52 days
Median : 1372.53 days
Min    : 656.56 days
Max    : 1947.17 days

Rows removed because of invalid values: 0


In [ ]:
# ============================================================
# FACTORY PREDICTION CHECK
# ============================================================

factory_check = []

for factory in factories:

    # Use model_df which contains all engineered features
    temp = model_df[model_features].copy()

    temp["Factory"] = factory

    predictions = model.predict(temp)

    factory_check.append({
        "Factory": factory,
        "Average Predicted Lead Time": predictions.mean(),
        "Median Predicted Lead Time": np.median(predictions),
        "Minimum Predicted Lead Time": predictions.min(),
        "Maximum Predicted Lead Time": predictions.max()
    })

factory_check_df = pd.DataFrame(factory_check)

factory_check_df = factory_check_df.sort_values(
    "Average Predicted Lead Time"
)

display(factory_check_df)

,Factory,Average Predicted Lead Time,Median Predicted Lead Time,Minimum Predicted Lead Time,Maximum Predicted Lead Time
0,Lot's O' Nuts,1485.855812,1539.035917,656.560000,1947.170000
4,The Other Factory,1486.084207,1539.589722,655.486667,1945.620000
2,Sugar Shack,1486.201781,1539.528778,655.486667,1945.620000
1,Wicked Choccy's,1486.210971,1538.805444,655.396667,1944.463333
3,Secret Factory,1487.403900,1541.293694,655.486667,1944.833333


In [ ]:
# ============================================================
# RECOMMENDATION DISTRIBUTION
# ============================================================

print("Recommendation Distribution")
print("--------------------------------")

print(
    best_factory_df["Recommendation"]
    .value_counts()
)

print("\nRecommended Factory Distribution")
print("--------------------------------")

print(
    best_factory_df[
        best_factory_df["Recommendation"] == "Reassign"
    ]["Recommended Factory"]
    .value_counts()
)

print("\nCurrent Factory Distribution")
print("--------------------------------")

print(
    best_factory_df["Current Factory"]
    .value_counts()
)

Recommendation Distribution
--------------------------------
Recommendation
Keep Current Factory    4386
Reassign                4163
Name: count, dtype: int64

Recommended Factory Distribution
--------------------------------
Recommended Factory
Wicked Choccy's      1602
Lot's O' Nuts        1213
The Other Factory     503
Secret Factory        465
Sugar Shack           380
Name: count, dtype: int64

Current Factory Distribution
--------------------------------
Current Factory
Lot's O' Nuts        4760
Wicked Choccy's      3445
Secret Factory        213
The Other Factory      98
Sugar Shack            33
Name: count, dtype: int64


In [ ]:
import joblib

model = joblib.load(
    "/content/best_model_random_forest.pkl"
)

print("Recommendation model loaded successfully!")
print(model)

Recommendation model loaded successfully!
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Product ID', 'Product Name',
                                                   'Division', 'Region',
                                                   'State/Province',
                                                   'Ship Mode', 'Factory']),
                                                 ('numerical',
                                                  Pipeline(steps=[('imputer',
                                            